In [1]:
###The following function is used for the Multirank change point method.

"""
###Multirank###
Multivariate change-point estimation
See functions :
autoDynKWRupt for automatic estimation of the number of change-points
DynMultiKW for displaying solutions for a range of values of the number of change-points

See also DynMultiLinear/autoDynMultiLinear for the linear,
max-likehood-based version (under the Gaussian assumption
and DynKernel/autoDynKernel for a kernel based version.

"""
import numpy
from numpy import searchsorted,empty,zeros,triu,arange,argmax,diag,eye,cov,dot,inf,sqrt,mean,exp,isnan,argmin
from numpy.random import permutation
from numpy.matlib import repmat
from multirankCP import multiRankChangePointDetect
from scipy.stats import chi2
from multirank import createCDFTables

import numpy
import numpy 
from numpy import zeros,sqrt
from multirankCP import multiRankChangePointDetect
from dynkw import autoDynKWRupt, DynMultiKW
from multirank import multirank

"""
Kruskal Wallis test is defined by:
K = 12/N/N+1 \sum_i = 1^s R_i^2/n_i -3(N+1)
where :
 N is sample size
 n_i size of segment_i
 R_i  =  R_i1+...+R_in_i
 s is number of segments (s-1 segments boundaries)
"""

def dynprog(matD,Kmax):
   """Marc Lavielle dcpc inspired
   """
   N,N  =  matD.shape
   I  =  repmat(-inf,Kmax,N)
   t  =  zeros((Kmax-1,N),dtype = int)

   I[0]  =  matD[0]

   for k in range(1,Kmax):
      for L in range(k,N):
         am  =  argmax(matD[1:L+1,L]+I[k-1,0:L])
         t[k-1,L]  =  am
         I[k,L]  =   (matD[1:L+1,L]+I[k-1,0:L])[am]

   J  =  I[:,N-1]#-3*(N+1) # with correction TO FIX

   t_est  =  (N-1)*eye(Kmax,dtype = int)
   # backward recursion

   for K in range(1,Kmax):
      for k in range(K-1,-1,-1):
         t_est[K,k]  =  t[k,t_est[K,k+1]]

   return J,t_est



# "contrast matrix", ie, for each 1< = n1< = n2< = N, calc

def multiRankMatrixx(cdf):
    """ returns an upper triangular array
    """
    nDim,winLength  =  cdf.shape
    M  =  zeros((winLength,winLength,nDim),dtype = float)
    N  =  zeros((winLength,winLength))

    R  =  zeros((winLength,winLength),dtype = float)
    
    ## =  = 
    a = cov(cdf)/(winLength*winLength)
    invSigma = 1/a
   #     invSigma  =  numpy.linalg.inv(a)
    
    for i in range(winLength):
       Ni  =  arange(1,winLength+1-i,dtype = float)
       N[i,i:] =  Ni
       M[i,i:] = (cdf[:,i:].cumsum(axis = 1).T/repmat(Ni,nDim,1).T) - winLength/2.



    for i in range(winLength):
       for j in range(i,winLength):
          R[i,j]  =  N[i,j]*dot(dot(M[i,j],invSigma),M[i,j])
    return R/(winLength*winLength)


def DynMultiKWW(x,Kmax  =  20):
   """ Dynamic programming + Multivariate KW
   
   Parameters
   ----------
   x : data, size dimension x sample length
   Kmax : maximum number of change points to be computed, default is Kmax  =  20
   
   Returns
   -------
   stats : value of the statistic for number of change from 1 to Kmax
   chgP  : Lower triangular matrix of size Kmax x Kmax, with position of the end of the k segments at line k
   """
   nDim,winLength  =  x.shape

   cdf,cdfComp  =  createCDFTables(x,x)

   matD  =  multiRankMatrixx(cdf)

   stats,chgP  =  dynprog(matD,Kmax)

   return stats,chgP
   

def multiRankMatrix(cdf):
    """ returns an upper triangular array
    """
    nDim,winLength  =  cdf.shape
    M  =  zeros((winLength,winLength,nDim),dtype = float)
    N  =  zeros((winLength,winLength))

    R  =  zeros((winLength,winLength),dtype = float)
    
    ## =  = 
    a = cov(cdf)/(winLength*winLength)
    invSigma  =  numpy.linalg.inv(a)
    
    for i in range(winLength):
       Ni  =  arange(1,winLength+1-i,dtype = float)
       N[i,i:] =  Ni
       M[i,i:] = (cdf[:,i:].cumsum(axis = 1).T/repmat(Ni,nDim,1).T) - winLength/2.



    for i in range(winLength):
       for j in range(i,winLength):
          R[i,j]  =  N[i,j]*dot(dot(M[i,j],invSigma),M[i,j])
    return R/(winLength*winLength)


def DynMultiKW(x,Kmax  =  20):
   """ Dynamic programming + Multivariate KW
   
   Parameters
   ----------
   x : data, size dimension x sample length
   Kmax : maximum number of change points to be computed, default is Kmax  =  20
   
   Returns
   -------
   stats : value of the statistic for number of change from 1 to Kmax
   chgP  : Lower triangular matrix of size Kmax x Kmax, with position of the end of the k segments at line k
   """
   nDim,winLength  =  x.shape

   cdf,cdfComp  =  createCDFTables(x,x)

   matD  =  multiRankMatrix(cdf)

   stats,chgP  =  dynprog(matD,Kmax)

   return stats,chgP
    
    

def multiKW(x,listCh):
   """
   Multivariate Kruskal Wallis
   x : data (nDim x winLength)
   chP : collection of changePoints
   Do not include 0 and winLength!
   """


   nDim,winLength  =  x.shape
   chP  =  list(listCh)
   
   chP.append(winLength)
   chP.append(0)
   chP.sort()
   
   
   cdf,cdfComp  =  createCDFTables(x,x)

   invSigma  =  numpy.linalg.inv(numpy.cov(cdf)/(winLength*winLength))

   H  =  0.
   
   for i in range(len(chP)-1):
      deb  =  chP[i]
      fin  =  chP[i+1]
      R  =  (cdf[:,deb:fin].sum(axis = 1)/(fin-deb))-winLength/2.

      H +=  (fin-deb)*dot(dot(R,invSigma),R)

   return H/(winLength*winLength)
   



############################
# kernel related functions #
############################
def grammat(X):
    '''Computes Gram matrix for Gaussian kernel with kernel bandwidth as suggested by the authors
    '''
    scaProd  =  dot(X.T,X)
    N,N  =  scaProd.shape
    SqNorm  =  zeros(N)
    for i in range(N):
        SqNorm[i]  =  (X[:,i]**2).sum()

    K  =  zeros(scaProd.shape)
    for i in range(N):
        for j in range(N):
            K[i,j] = SqNorm[i]+SqNorm[j]-2*scaProd[i,j]

    empVar  =  sqrt(((X.T-mean(X,axis = 1))**2).sum(axis = 1).mean())

    sig  =  1.06 * empVar * N**(1./5)

    #med  =  median(K[K>1e-9])

    return exp(-K/(2*sig*sig))


def kernelIntraScatterFast(K):
    '''Computes kernel intra scatter matrix.
    Argument : K gram matrix
    '''
    N,N  =  K.shape
    matD  =  eye(N)

    Ni  =  eye(N)

    for i in range(N-1):
        for j in range(i+1,N):
            Ni[i,j]  =  j-i+1
            matD[i,j] = (matD[i,j-1]+2*K[j,i:j+1].sum()-K[j,j])

    R  =  Ni-matD/Ni ## Ni-, assuming ones on grammat diagonal
    R[isnan(R)] = 0.
    return R


def dynprogPos(matD,Kmax):
   """Marc Lavielle dcpc inspired
   """
   N,N  =  matD.shape
   I  =  repmat(inf,Kmax,N)
   t  =  zeros((Kmax-1,N),dtype = int)

   I[0]  =  matD[0]

   for k in range(1,Kmax):
      for L in range(k,N):
         am  =  argmin(matD[1:L+1,L]+I[k-1,0:L])
         t[k-1,L]  =  am
         I[k,L]  =   (matD[1:L+1,L]+I[k-1,0:L])[am]

   J  =  I[:,N-1]#-3*(N+1) # with correction TO FIX

   t_est  =  (N-1)*eye(Kmax,dtype = int)
   # backward recursion

   for K in range(1,Kmax):
      for k in range(K-1,-1,-1):
         t_est[K,k]  =  t[k,t_est[K,k+1]]

   return J,t_est


def DynKernel(X,Kmax):
   """
   Dynamic programming + kernel
   """   
   nDim,winLength  =  X.shape
   K  =  grammat(X)
   matD  =  kernelIntraScatterFast(K)

   stats,chgP  =  dynprogPos(matD,Kmax)

   return stats,chgP
   
#################
# Linear method #
#################

def matContLinear(X):


   nDim,winLength  =  X.shape
   Xc  =  (X.T-X.mean(axis = 1)).T
   invSigma  =  numpy.linalg.inv(cov(X))
   
   R  =  zeros((winLength,winLength),dtype = float)

   M  =  zeros((winLength,winLength,nDim),dtype = float)
   N  =  zeros((winLength,winLength))

   for i in range(winLength):
      Ni  =  arange(1,winLength+1-i,dtype = float)
      N[i,i:] =  Ni
      M[i,i:] = (Xc[:,i:].cumsum(axis = 1).T/repmat(Ni,nDim,1).T)

   for i in range(winLength):
      for j in range(i,winLength):
         R[i,j]  =  N[i,j]*dot(dot(M[i,j],invSigma),M[i,j])

   return R


def DynMultiLinear(X,Kmax):
   """
   Dynamic programming + linear test
   """
   nDim,winLength  =  X.shape
  
   matD  =  matContLinear(X)

   stats,chgP  =  dynprog(matD,Kmax)

   return stats,chgP
      
   

def vostri(x,alpha = .01):
   # Multi Rank + Vostrikova
   nDim,winLength  =  x.shape
   changePoints  =  []

   def recurSeg(X,C,deb,fin,alpha):
      
      maxVal,maxPos,pval,nLib,stats  =  multiRankChangePointDetect(X[:,deb:fin],C[:,deb:fin])
      if pval < alpha:
         changePoints.append(maxPos+deb)
         recurSeg(X,C,deb,deb+maxPos,alpha)
         recurSeg(X,C,deb+maxPos,fin,alpha)

   recurSeg(x,numpy.ones(x.shape),0,winLength,alpha)

   return sorted(changePoints)


def rupturepente(leastPC):
   Y  =  leastPC
   l  =  leastPC.size
   x  =  arange(l)

   aL_  =  []
   bL_  =  []
   aR_  =  []
   bR_  =  []

   S  =  []

   for k in range(1,l-1):
      Yleft  =  Y[:k+1]
      Xleft  =  arange(0,k+1)

      aL  =  (mean(Yleft*Xleft)-mean(Yleft)*mean(Xleft))/(mean(Xleft**2.)-(mean(Xleft))**2.)
      bL  =  mean(Yleft)-aL*mean(Xleft)

      aL_.append(aL)
      bL_.append(bL)
      

      Yright  =  Y[k:]
      Xright  =  arange(k,l)

      aR  =  (mean(Yright*Xright)-mean(Yright)*mean(Xright))/(mean(Xright**2.)-(mean(Xright))**2.)
      bR  =  mean(Yright)-aR*mean(Xright)

      aR_.append(aR)
      bR_.append(bR)


      S.append( sum((Yleft-aL*Xleft-bL)**2.)+sum((Yright-aR*Xright-bR)**2.) )

   am  =  argmin(S)
      
   return am+1 #+1 for k started at 1 !



def autoDynKernel(x,Kmax = 20):
    nDim,winLength  =  x.shape
    stats,chP  =  DynKernel(x,Kmax)
    numCh  =  rupturepente(stats)
    return numCh,chP[numCh]
   

def autoDynMultiLinear(x,Kmax = 20):
    nDim,winLength  =  x.shape
    stats,chP  =  DynMultiLinear(x,Kmax)
    numCh  =  rupturepente(stats)
    return numCh,chP[numCh]

def autoDynKWRupt(x,Kmax = 20):
    '''use Dynamic programming Kruskal-Wallis, and automagically returns number of changes using rupturepente heuristic algorithm
    Parameters
    ----------
    x : numpy array of data (nDim x windowLength)
    Kmax : (optional, default to 20) : number of change point to compute before applying slope heuristic

    Returns
    -------
    numCh : number of change points
    chP : array of the position of the end of the segment boundaries for the values that are different to 0.
    '''
    nDim,winLength  =  x.shape
    stats,chP  =  DynMultiKW(x,Kmax)
    
    # test on stats[1] to check for homogeneity
    if stats[1]< chi2(nDim).ppf(0.9999):
        return 0,chP[0]
    else:
        numCh  =  rupturepente(stats)
        
        return numCh,chP[numCh]

def autoDynKWRuptt(x,Kmax = 20):
    '''use Dynamic programming Kruskal-Wallis, and automagically returns number of changes using rupturepente heuristic algorithm
    Parameters
    ----------
    x : numpy array of data (nDim x windowLength)
    Kmax : (optional, default to 20) : number of change point to compute before applying slope heuristic

    Returns
    -------
    numCh : number of change points
    chP : array of the position of the end of the segment boundaries for the values that are different to 0.
    '''
    nDim,winLength  =  x.shape
    stats,chP  =  DynMultiKWW(x,Kmax)

    # test on stats[1] to check for homogeneity
    if stats[1]< chi2(nDim).ppf(0.9999):
        return 0,chP[0]
    else:
        numCh  =  rupturepente(stats)

        return numCh,chP[numCh]




<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
C:\Users\admin\AppData\Local\Temp\ipykernel_29228\1158534488.py:30: SyntaxWarning: invalid escape sequence '\s'
  """


In [12]:
###Loading data
import numpy as np
from scipy.stats import multivariate_normal
import ruptures as rpt
import math
import os
import random
#### Basic parameter settings
n = 800              # number of samples
px = 100             # data dimension
times = 1000           # number of experiments
an = math.floor(n ** 0.5)  # tuning parameter alpha_n
type = "ba"          # "ba" for balanced, "im" for imbalanced setting

# Construct the filename and full path based on `type`, and `px`
file_name = f"e3{type}{px}.npy"
data_path = os.path.join("E:/ckpca_code/change/simu_data", file_name)

# Check if the file exists
if not os.path.exists(data_path):
    raise FileNotFoundError(f"File not found: {data_path}")

# Load the data
flat = np.load(data_path)
print("flat shape:", flat.shape)

# Reshape into (times, px, n)
recovered = flat.reshape((times, px, n))
print("recovered shape:", recovered.shape)

##set.seed
SEED  =  1111
os.environ['PYTHONHASHSEED']  =  str(SEED)
random.seed(SEED)
np.random.seed(SEED)

###Save the change point locations and the number 
###of change points for various methods
numCh1 = np.zeros(times);chP1 = np.zeros((20,times))
numCh2 = np.zeros(times);chP2 = np.zeros((20,times))
numCh3 = np.zeros(times);chP3 = np.zeros((20,times))
numCh4 = np.zeros(times);chP4 = np.zeros((20,times))
numCh5 = np.zeros(times);chP5 = np.zeros((20,times))
numCh6 = np.zeros(times);chP6 = np.zeros((20,times))
numCh7 = np.zeros(times);chP7 = np.zeros((20,times))
numCh8 = np.zeros(times);chP8 = np.zeros((20,times))

##balanced or imbalanced
if type  ==  "ba":
    n1, n2, n3, n4  =  100, 200, 300, 400
    n5, n6, n7, n8  =  500, 600, 700, 800
elif type  ==  "im":
    n1, n2, n3, n4  =  30, 170, 350, 440
    n5, n6, n7, n8  =  520, 630, 710, 800
else:
    raise ValueError("type must be either 'ba' or 'im'")

In [ ]:
!pip install ruptures

In [ ]:
%%time

###change point detection

for kkk in range(times):  
    print(f"Running iteration {kkk + 1} of {times}")
    ##loading data
    x1 = recovered[kkk - 1,:, : ]
    x1u = x1.reshape(px, n)
    x = x1u.T

    ##cpca
    ##dimension reduction
    Mx = np.cov(x, rowvar=False) * n / (n - 1)
    sigma = np.zeros((px, px))
    zs = int(np.floor(n/an))
    for j in range(1, zs):
        sigma += np.cov(x[((j-1)*an):an*j, :], rowvar=False)
    sigma += np.cov(x[((zs-1)*an):n, :], rowvar=False)
    sigma /= zs
    Dx = Mx - sigma
    va, evector = np.linalg.eig(Dx)
    evalue = np.abs(va)
    evalue_full = np.zeros(px+1)
    evalue_full[:px] = evalue
    c1n = math.log(math.log(n)) * math.sqrt(px / n) / 5
    c2n = math.log(n) * np.mean(evalue[int(math.sqrt(n)/2):int(math.sqrt(n))])
    ccn = max(c1n, c2n)
    evalueplus = evalue + ccn
    lambda1 = evalueplus[1:px] / evalueplus[:-1] 
    lambda2 = np.where(lambda1[:-1] < 0.5)[0] 
    aaa = np.min(lambda1[:-1])
    if aaa > 0.5:
        hatq = 1
    else:
        hatq = max(lambda2)
    B1 = evector[:, :2]
    B = evector[:, :hatq]
    B_norm = np.sqrt(np.sum(B**2, axis=0))
    B /= B_norm
    cc1 = np.dot(B.T, x.T)
    
    ## PCA method
    Dx = Mx
    spec = np.linalg.eig(Dx)
    evalue = np.zeros(px+1)
    va = np.abs(spec[0])
    evalue[1:px+1] = va
    evector = spec[1]
    total = np.sum(evalue)
    to = 0.8 * total
    cto = np.zeros(px)
    for i in range(px):
        for j in range(i+1):
            cto[i] += evalue[j]
    lo = np.where(cto > to)[0]
    hatq = lo[0]
    B = evector[:, :hatq]
    cc2 = np.dot(B.T, x.T)

    ## mahalanobis distance
    import math
    tau = 0.5
    alpha_n = int(sqrt(n))
    a = zeros((n*n, px))
    Mx = zeros((px, px))
    for i in range(px):
        mt = numpy.mat(x[:, i])
        tt = numpy.matmul(numpy.transpose(mt), numpy.ones((1, n))) - numpy.matmul(numpy.ones((n, 1)), mt)
        tx = tt.reshape(1, n*n)
        a[:, i] = tx
    Mx = numpy.matmul(numpy.transpose(a), a) / (n * (n - 1))
    sigma = zeros((px, px))
    for j in range(int(n / alpha_n)):
        aaa = numpy.transpose(x[(j * alpha_n):((j + 1) * alpha_n - 1), :])
        sigma = sigma + numpy.cov(aaa)
    sigma = sigma / n * alpha_n
    Dx = Mx - 2 * sigma
    evalue, evector = numpy.linalg.eig(Dx)
    evalue = numpy.sort(evalue)[::-1]
    ccn = 0.5 * math.log(math.log(n)) * sqrt(px / n)
    evalueplus = evalue + ccn
    lambda1 = numpy.zeros(px)
    for i in range(px - 1):
        lambda1[i] = evalueplus[i + 1] / evalueplus[i]
    a = numpy.where(lambda1[0:px - 1] < 0.5)
    if (numpy.min(lambda1[0:px - 1]) > 0.5):
        hatq = 0
    else:
        hatq = numpy.max(a)
    B = evector[:, hatq]
    cc = numpy.matmul(numpy.transpose(B), numpy.transpose(x))
    cc3 = numpy.array(numpy.mat(cc))


    ##Raw data
    cc4 = x
    
    # ##multirank after CKPCA
    numCh1[kkk],chP11  =  autoDynKWRuptt(cc1)
    if (numCh1[kkk]>0):
        chP1[0:(int(numCh1[kkk])),kkk] = chP11[chP11!= 0][0:(int(numCh1[kkk]))]
    print(numCh1[kkk])
    print(chP1[0:(int(numCh1[kkk])),kkk])

    ##multirank after KPCA
    numCh2[kkk],chP22  =  autoDynKWRupt(cc2)
    if (numCh2[kkk]>0):
        chP2[0:(int(numCh2[kkk])),kkk] = chP22[chP22!= 0][0:(int(numCh2[kkk]))]
    print(numCh2[kkk])
    print(chP2[0:(int(numCh2[kkk])),kkk])
    
    ##multirank after mahalanobis distance
    numCh3[kkk],chP33  =  autoDynKWRuptt(cc3)
    if (numCh3[kkk]>0):
        chP3[0:(int(numCh3[kkk])),kkk] = chP33[chP33!= 0][0:(int(numCh3[kkk]))]
    print(numCh3[kkk])
    print(chP3[0:(int(numCh3[kkk])),kkk])

    ##multirank without dimension reduction
    numCh4[kkk],chP44  =  autoDynKWRupt(numpy.transpose(cc4))
    if (numCh4[kkk]>0):
        chP4[0:(int(numCh4[kkk])),kkk] = chP44[chP44!= 0][0:(int(numCh4[kkk]))]
    print(numCh4[kkk])
    print(chP4[0:(int(numCh4[kkk])),kkk])
    
    ##
    cc1u  =  cc1.T
    model1  =  rpt.KernelCPD(kernel = "rbf", min_size = 30).fit(cc1u)
    pen = 3
    chP55  =  model1.predict(pen = pen)  
    numCh5[kkk]  =  len(chP55)-1  
    chP55 = np.array(chP55)
    if (numCh5[kkk]>0):
        chP5[0:(int(numCh5[kkk])),kkk] = chP55[chP55!= 0][0:(int(numCh5[kkk]))]
    
    print(numCh5[kkk])
    print(chP5[0:(int(numCh5[kkk])),kkk])

    ##
    cc2u  =  cc2.T  
    model2  =  rpt.KernelCPD(kernel = "rbf", min_size = 30).fit(cc2u)
    pen  =  3
    chP66  =  model2.predict(pen = pen)  
    numCh6[kkk]  =  len(chP66)-1 
    chP66 = np.array(chP66)
    if numCh6[kkk] > 0:
        chP6[0:int(numCh6[kkk]), kkk]  =  chP66[chP66 !=  0][0:int(numCh6[kkk])]
    
    print(numCh6[kkk])
    print(chP6[0:int(numCh6[kkk]), kkk])

    cc3u  =  cc3.T 
    model3  =  rpt.KernelCPD(kernel = "rbf", min_size = 30).fit(cc3u)
    pen  =  3
    chP77  =  model3.predict(pen = pen)
    numCh7[kkk]  =  len(chP77)-1
    chP77 = np.array(chP77)
    if numCh7[kkk] > 0:
        chP7[0:int(numCh7[kkk]), kkk]  =  chP77[chP77 !=  0][0:int(numCh7[kkk])]
    
    print(numCh7[kkk])
    print(chP7[0:int(numCh7[kkk]), kkk])
    
    cc4u  =  cc4.T
    model4  =  rpt.KernelCPD(kernel = "rbf", min_size = 30).fit(cc4u)
    pen  =  3
    chP88  =  model4.predict(pen = pen)
    numCh8[kkk]  =  len(chP88)-1
    chP88 = np.array(chP88)
    if numCh8[kkk] > 0:
        chP8[0:int(numCh8[kkk]), kkk]  =  chP88[chP88 !=  0][0:int(numCh8[kkk])]
    
    print(numCh8[kkk])
    print(chP8[0:int(numCh8[kkk]), kkk])

import time
time.sleep(2)  

In [27]:
import numpy as np

def randi(posi, num, n, times, n1, n2, n3, n4, n5, n6, n7, n8):
    """
    Calculate Rand Index with quantile confidence intervals.
    
    Parameters:
    - posi: 2D numpy array, change point positions for each simulation (shape: times x max_num)
    - num: 1D numpy array, number of change points detected in each simulation (length: times)
    - n: int, length of sequence
    - times: int, number of simulations
    - n1,...,n8: int, boundary indices defining true clusters (for ground truth)
    
    Returns:
    - mean_ri: float, mean Rand Index over all simulations
    - q025: float, 2.5% quantile of Rand Index
    - q975: float, 97.5% quantile of Rand Index
    """

    # Convert inputs to integers for indexing
    numberr1  =  num.astype(int)
    positionn1  =  posi.astype(int)

    # Ground truth cluster assignment based on boundaries n1..n8
    index0  =  np.zeros(n, dtype = int)
    boundaries  =  [n1, n2, n3, n4, n5, n6, n7, n8]
    cluster_label  =  1
    start  =  0
    for end in boundaries:
        end  =  int(end)
        if end > start:
            index0[start:end]  =  cluster_label
            cluster_label +=  1
            start  =  end

    # Precompute total possible pairs
    all_pairs  =  n * (n - 1) // 2
    if all_pairs  ==  0:
        # Avoid division by zero for trivial cases
        return 0.0, 0.0, 0.0

    randindex  =  np.zeros(times)

    for j in range(times):
        index  =  np.zeros(n, dtype = int)  # Cluster labels for simulation j
        num_j  =  int(numberr1[j]) if j < len(numberr1) else 0

        # Assign clusters based on detected change points
        if num_j  ==  0:
            # Only one cluster for all points
            index[:]  =  1
        elif num_j  ==  1:
            # Two clusters split at first detected change point
            pos_start  =  int(positionn1[j, 0]) if (j < positionn1.shape[0] and positionn1.shape[1] > 0) else 0
            pos_start  =  max(0, min(pos_start, n))
            index[0:pos_start]  =  1
            index[pos_start:n]  =  2
        else:
            # Multiple clusters
            if j < positionn1.shape[0] and positionn1.shape[1] > 0:
                # First cluster
                pos_start  =  int(positionn1[j, 0])
                pos_start  =  max(0, min(pos_start, n))
                index[0:pos_start]  =  1

                # Intermediate clusters
                for i in range(min(num_j - 1, positionn1.shape[1] - 1)):
                    pos1  =  int(positionn1[j, i])
                    pos2  =  int(positionn1[j, i + 1])
                    pos1  =  max(0, min(pos1, n))
                    pos2  =  max(0, min(pos2, n))
                    if pos1 < pos2:
                        index[pos1:pos2]  =  i + 2

                # Last cluster
                pos_end  =  int(positionn1[j, num_j - 1]) if num_j > 0 else pos_start
                pos_end  =  max(0, min(pos_end, n))
                if pos_end < n:
                    index[pos_end:n]  =  num_j + 1

        # Calculate Rand Index components (true positive + true negative pairs)
        tp  =  0
        tn  =  0
        for jj in range(1, n):
            for hh in range(jj):
                same_cluster_pred  =  (index[jj] ==  index[hh])
                same_cluster_true  =  (index0[jj]  ==  index0[hh])
                if same_cluster_pred and same_cluster_true:
                    tp +=  1
                elif not same_cluster_pred and not same_cluster_true:
                    tn +=  1

        randindex[j]  =  (tp + tn) / all_pairs

    # Compute mean and 95% confidence interval from quantiles
    mean_ri  =  np.mean(randindex)
    q025, q975  =  np.quantile(randindex, [0.025, 0.975])

    return mean_ri, q025, q975


def batch_apply_randi():
    """Batch process all chP arrays with error handling and quantile reporting"""
    chp_list  =  [chP1, chP2, chP3, chP4, chP5, chP6, chP7, chP8]
    numch_list  =  [numCh1, numCh2, numCh3, numCh4, numCh5, numCh6, numCh7, numCh8]
    
    randi_results  =  []

    for i, (chp, numch) in enumerate(zip(chp_list, numch_list)):
        try:
            mean_ri, q025, q975  =  randi(chp.T, numch, n, times, n1, n2, n3, n4, n5, n6, n7, n8)
            
            result  =  {
                'mean': mean_ri,
                'q025': q025,
                'q975': q975
            }
            
            randi_results.append(result)
            
            print(f"  chP{i+1} Rand Index:")
            print(f"    Mean: {mean_ri:.4f}")
            print(f"    2.5% quantile: {q025:.4f}")
            print(f"    97.5% quantile: {q975:.4f}")
        
        except Exception as e:
            print(f"  Error processing chP{i+1}: {e}")
            randi_results.append(None)
        
        print()  # Add empty line for readability
    
    return randi_results


In [ ]:
mk  =  np.zeros(8)
mk[0]  =  np.mean(numCh1)
mk[1]  =  np.mean(numCh2)
mk[2]  =  np.mean(numCh3)
mk[3]  =  np.mean(numCh4)
mk[4]  =  np.mean(numCh5)
mk[5]  =  np.mean(numCh6)
mk[6]  =  np.mean(numCh7)
mk[7]  =  np.mean(numCh8)

me_np  =  np.zeros(8)
me_np[0]  =  np.mean((numCh1 - 7) ** 2)
me_np[1]  =  np.mean((numCh2 - 7) ** 2)
me_np[2]  =  np.mean((numCh3 - 7) ** 2)
me_np[3]  =  np.mean((numCh4 - 7) ** 2)
me_np[4]  =  np.mean((numCh5 - 7) ** 2)
me_np[5]  =  np.mean((numCh6 - 7) ** 2)
me_np[6]  =  np.mean((numCh7 - 7) ** 2)
me_np[7]  =  np.mean((numCh8 - 7) ** 2)

print("mk  = ", mk)
print("me  = ", me_np)

In [ ]:
results  =  batch_apply_randi()

In [44]:
# Extract values from results (which is a list of dictionaries)
# Use np.nan for entries that failed (i.e., are None)
mean_vals  =  np.array([res['mean'] if res is not None else np.nan for res in results])
q025_vals  =  np.array([res['q025'] if res is not None else np.nan for res in results])
q975_vals  =  np.array([res['q975'] if res is not None else np.nan for res in results])

# Combine all arrays into a single matrix for saving
# Each row: [mk, me_np, mean, q025, q975]
combined  =  np.column_stack((mk, me_np, mean_vals, q025_vals, q975_vals))

# Define the output file path
csv_file_name = f"pysimue3{type}{px}.csv"
csv_path = os.path.join("E:/ckpca_code", csv_file_name)

# Save the matrix to CSV with 6 decimal places
np.savetxt(csv_path, combined, delimiter = ",", fmt = "%.6f")


